# VERA v2 — M4 detector-box temporal campaign

Run after the detector-box M3 faithful checkpoint has been saved to Drive. The notebook has separate grid/paper/final switches so a Kaggle session can stop and resume without changing the protocol.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, zipfile
REPO_URL='https://github.com/hiennguyendang/phase_2_3_4_5.git'
def bootstrap_repo():
    target=Path('/kaggle/working/vera_repo')
    if (target/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): return target
    if target.exists(): shutil.rmtree(target)
    try:
        subprocess.run(['git','clone',REPO_URL,str(target)],check=True)
        print('source: GitHub commit',subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip()); return target
    except Exception as exc: print('[fallback] GitHub clone unavailable:',exc)
    root=Path('/kaggle/input/datasets/nguynnghin/vera-v2-code'); archives=list(root.glob('*.zip')) if root.exists() else []; candidates=[root]
    if len(archives)==1:
        extracted=Path('/kaggle/working/vera_v2_code_extracted'); extracted.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archives[0]) as zf: zf.extractall(extracted)
        candidates=[extracted]+list(extracted.glob('*/'))
    for candidate in candidates:
        if (candidate/'phase_2/scripts/yolo/5-infer_yolo.py').exists(): shutil.copytree(candidate,target,dirs_exist_ok=True); print('source: Kaggle code dataset',candidate); return target
    raise RuntimeError('GitHub unavailable; attach /kaggle/input/datasets/nguynnghin/vera-v2-code')
REPO_DIR=bootstrap_repo()
sys.path.insert(0,str(REPO_DIR/'kaggle_notebooks'))
from vera_common import find_bundle, find_m2_outputs, FEATURE_ROOT, configure_drive, copy_tree
bundle=find_bundle(); m2_output=find_m2_outputs(); remote=configure_drive()
print('bundle:',bundle,'detector output:',m2_output)

In [ ]:
# Assemble writable M3/M4 inputs and pull both M3 faithful/oracle checkpoints.
labels=Path('/kaggle/working/m3_labels_detector_v2')
if labels.exists(): shutil.rmtree(labels)
copy_tree(bundle/'m3_labels_base',labels)
m2=m2_output/'m3_labels_detector_v2'
for name in ['boxes_det.npy','present_mask_det.npy','detector_provenance.json']: shutil.copy2(m2/name,labels/name)
m4labels=Path('/kaggle/working/m4_labels')
if m4labels.exists(): shutil.rmtree(m4labels)
copy_tree(bundle/'m4_labels',m4labels)
m3dir=Path('/kaggle/working/m3_main'); gt_dir=Path('/kaggle/working/m3_gt')
for d in [m3dir,gt_dir]:
    if d.exists(): shutil.rmtree(d)
subprocess.run(['rclone','copy',remote+'/m3_runs/m3v2_vera_graph_lse_det',str(m3dir)],check=True)
subprocess.run(['rclone','copy',remote+'/m3_runs/m3v2_vera_graph_lse_gt',str(gt_dir)],check=True)
assert (m3dir/'best.pt').exists() and (gt_dir/'best.pt').exists()
print('M3 main:',m3dir/'best.pt'); print('M3 GT oracle:',gt_dir/'best.pt')

In [ ]:
# Shared M4 configuration. Set one of the switches to 0 when resuming later.
RUN_GRID=1; RUN_PAPER=1; RUN_FINAL=1
env=os.environ.copy(); env.update(PY='python',DEVICE='cuda:0',BATCH='8',W='2',EVAL_W='2',EP='40',
    M3_CKPT=str(m3dir/'best.pt'), M3_GT_CKPT=str(gt_dir/'best.pt'), M3LAB=str(labels),
    M4LAB=str(m4labels), PAIRS=str(m4labels/'m3_pairs.jsonl'), FEAT=str(FEATURE_ROOT),
    CACHE_DET='/kaggle/working/m4_region_cache_m3v2_detector', CACHE_GT='/kaggle/working/m4_region_cache_m3v2_gt_oracle',
    RUNS='/kaggle/working/m4_runs', LOGDIR='/kaggle/working/m4_logs', DIAGDIR='/kaggle/working/m4_diagnostics',
    SELECTED_ENV='/kaggle/working/m4_diagnostics/selected_coefficients.env',
    MS_CSV=str(m4labels/'MS_CXR_T_temporal_image_classification_v1.0.0.csv'), SYNC_REMOTE=remote+'/m4_runs', SYNC_EVERY='0')
subprocess.run(['rclone','copy',remote+'/m4_diagnostics','/kaggle/working/m4_diagnostics'],check=False)
def run_rows(scope,names,bootstrap_cache=True):
    names=list(names)
    if bootstrap_cache and names:
        e=env.copy(); e.update(DEVICE='cuda:0',RUN_NAME=names.pop(0)); e.pop('SKIP_CACHE',None)
        subprocess.run(['bash',str(REPO_DIR/'phase_4/run_paper_m4_v2.sh'),'--profile','local4060','--scope',scope],cwd=REPO_DIR,env=e,check=True)
    for start in range(0,len(names),2):
        procs=[]
        for gpu,name in enumerate(names[start:start+2]):
            e=env.copy(); e.update(DEVICE=f'cuda:{gpu}',RUN_NAME=name,SKIP_CACHE='1')
            procs.append((name,subprocess.Popen(['bash',str(REPO_DIR/'phase_4/run_paper_m4_v2.sh'),'--profile','local4060','--scope',scope],cwd=REPO_DIR,env=e)))
        for name,p in procs: assert p.wait()==0, f'{name} failed'
subprocess.run(['bash',str(REPO_DIR/'phase_4/run_paper_m4_v2.sh'),'--profile','local4060','--scope','preflight'],cwd=REPO_DIR,env=env,check=True)

In [ ]:
if RUN_GRID:
    grid_names=[f'm4v2_grid_kl{kl}_dist{dist}_det' for kl in ['0025','0050','0075'] for dist in ['0350','0500','0650']]
    run_rows('grid',grid_names,bootstrap_cache=True)
    # Unfiltered pass only verifies/skips completed rows and selects coefficients from all nine validation runs.
    e=env.copy(); e['SKIP_CACHE']='1'; e.pop('RUN_NAME',None)
    subprocess.run(['bash',str(REPO_DIR/'phase_4/run_paper_m4_v2.sh'),'--profile','local4060','--scope','grid'],cwd=REPO_DIR,env=e,check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_runs',remote+'/m4_runs'],check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_diagnostics',remote+'/m4_diagnostics'],check=True)
    print('M4 coefficient grid saved; selected coefficients are locked in selected_coefficients.env')

In [ ]:
if RUN_PAPER:
    selected={}
    for line in Path(env['SELECTED_ENV']).read_text().splitlines():
        if '=' in line: k,v=line.split('=',1); selected[k]=v
    def tag(v): return f'{float(v):.3f}'.replace('.','')
    ktag,dtag=tag(selected['KL_WEIGHT']),tag(selected['DIST_WEIGHT']); suffix=f'kl{ktag}_dist{dtag}'
    paper_names=[selected['MAIN_RUN'],f'm4v2_regiondiff_{suffix}_det',f'm4v2_tempfuse_{suffix}_det',
      'm4v2_reg_base_det',f'm4v2_reg_kl{ktag}_det',f'm4v2_reg_dist{dtag}_det','m4v2_reg_smooth005_det',
      f'm4v2_reg_smooth005_kl{ktag}_det',f'm4v2_reg_smooth005_dist{dtag}_det',
      f'm4v2_reg_smooth005_{suffix}_det',f'm4v2_vera_{suffix}_gt_oracle']
    env['SKIP_DET_CACHE']='1'  # detector cache already came from the grid; bootstrap only the GT oracle cache
    run_rows('paper',paper_names,bootstrap_cache=True)
    env.pop('SKIP_DET_CACHE',None)
    subprocess.run(['rclone','copy','/kaggle/working/m4_runs',remote+'/m4_runs'],check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_diagnostics',remote+'/m4_diagnostics'],check=True)
    print('M4 paper ablations saved')

In [ ]:
if RUN_FINAL:
    e=env.copy()
    if (Path(env['CACHE_DET'])/'.m3_source.json').exists(): e['SKIP_CACHE']='1'
    subprocess.run(['bash',str(REPO_DIR/'phase_4/run_paper_m4_v2.sh'),'--profile','local4060','--scope','final'],cwd=REPO_DIR,env=e,check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_runs',remote+'/m4_runs'],check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_region_cache_m3v2_detector',remote+'/m4_region_cache_m3v2_detector'],check=True)
    subprocess.run(['rclone','copy','/kaggle/working/m4_diagnostics',remote+'/m4_diagnostics'],check=True)
    print('M4 final test, MS-CXR-T, inference, and temporal consistency saved')